In [15]:
import numpy as np
from scipy.linalg import expm as dense_expm
from scipy.linalg import norm as dense_norm
from scipy.sparse import csr_matrix, eye
from scipy.sparse.linalg import norm as sparse_norm
from tqdm.auto import tqdm

from exact import ising_hamiltonian_sparse, total_Z_magnetization_sparse

# 这个 notebook 里的“exact”更准确地说是 high-precision reference：
# 它使用一阶 Taylor step (I - i H t / r) 迭代 many steps，并在每步后归一化，
# 作为大系统下可计算的高精度参考值，而不是严格意义上的 exp(-iHt) 精确演化。
# 下面会额外加入一个小规模 dense expm 的 sanity check 来验证这种 reference 的可靠性。

# 参数
n = 10     # 自旋数
J = 0.5     # 相互作用强度
h = 1.0     # 横场强度
t = 1     # 演化时间

H1, H2, H3, H = ising_hamiltonian_sparse(n, J, h)

In [16]:
# 先看一下当前参考问题的规模，而不是直接把 sparse expm 跑出来
print(f"Hamiltonian shape: {H.shape}")
print(f"Hamiltonian number of non-zero elements: {H.nnz}")
print("This notebook computes a Taylor-step reference, then cross-checks it against dense expm on a smaller system.")

Hamiltonian shape: (1024, 1024)
Hamiltonian number of non-zero elements: 11264
This notebook computes a Taylor-step reference, then cross-checks it against dense expm on a smaller system.


In [17]:
r = 1000
# 1. 构造稀疏单位矩阵 I
I_sparse = eye(2**n, dtype=complex)

Taylor_1 = I_sparse - 1j * H * (t/r)

# 1. 定义数据 (data)、行索引 (row_ind) 和列索引 (col_ind)
# 这是一个列向量，形状为 (N, 1)
data = [1.0] 
row_ind = [0]  # 非零元素在第 0 行
col_ind = [0]  # 列向量只有 0 列

# 2. 使用 csr_matrix 构造稀疏列向量 (ket 态)
# 推荐使用 coo_matrix 构造后转为 csr_matrix，因为它在构建时更灵活，
# 且 csr_matrix 在后续矩阵向量乘法中性能更好。
initial_state_sparse = csr_matrix((data, (row_ind, col_ind)), shape=(2**n, 1), dtype=complex)
# print(f"开始计算稀疏矩阵 H 的 {r} 次方...")

# 初始化结果 U_approx 为稀疏单位矩阵
state = initial_state_sparse

# 循环进行稀疏矩阵乘法
# 警告：即使是稀疏矩阵乘法，1000 次乘法在大维度下仍然可能非常耗时，
# 并且结果 U_approx 会逐渐变得稠密。
for i in tqdm(range(r), desc="evolution steps"):
    state = Taylor_1 @ state
    # 一阶 Taylor step 不是严格幺正的，因此每一步都做归一化，
    # 用它构造一个大系统下稳定可用的 high-precision reference。
    state = state / sparse_norm(state)
    # 可以在这里添加检查点，看看矩阵是否变得太稠密
    # if (i + 1) % 100 == 0:
    #     print(f"完成第 {i + 1} 步，当前 U_approx 的非零元素数量: {U_approx.nnz}")

# print(f"\n✅ 稀疏矩阵的 {r} 次方计算完成！")
# print(f"最终 U_approx 的非零元素数量: {U_approx.nnz}")

evolution steps: 100%|██████████| 1000/1000 [00:00<00:00, 5636.56it/s]


In [18]:
sparse_norm(state)

np.float64(0.9999999999999998)

In [19]:
# Import the observable builder from exact.py as well so the notebook stays
# consistent with the script version.
O_sparse = total_Z_magnetization_sparse(n)

In [20]:
# 计算观测量期望值
# state = U_approx.dot(initial_state_sparse)
expectation_value = (state.getH().dot(O_sparse).dot(state)).toarray()[0,0]
expectation_value

np.complex128(-1.855094505512884+6.5483073150258e-18j)

In [23]:
# 小规模 sanity check: 用 dense expm 和 Taylor-step reference 做对照
n_check = 8
r_check = 1000000

_, _, _, H_check_sparse = ising_hamiltonian_sparse(n_check, J, h)
H_check_dense = H_check_sparse.toarray()
O_check_dense = total_Z_magnetization_sparse(n_check).toarray()

psi0_dense = np.zeros(2**n_check, dtype=complex)
psi0_dense[0] = 1.0

# 严格意义上的小系统精确演化
U_exact = dense_expm(-1j * H_check_dense * t)
psi_exact = U_exact @ psi0_dense
psi_exact = psi_exact / dense_norm(psi_exact)
exact_expectation = np.vdot(psi_exact, O_check_dense @ psi_exact)

# notebook 主流程对应的 high-precision reference
I_check = eye(2**n_check, dtype=complex)
taylor_step = I_check - 1j * H_check_sparse * (t / r_check)
state_check = csr_matrix(([1.0], ([0], [0])), shape=(2**n_check, 1), dtype=complex)
for _ in tqdm(range(r_check), desc='dense sanity check'):
    state_check = taylor_step @ state_check
    state_check = state_check / sparse_norm(state_check)

psi_reference = state_check.toarray().ravel()
reference_expectation = np.vdot(psi_reference, O_check_dense @ psi_reference)

print(f'n_check = {n_check}, r_check = {r_check}')
print(f'dense exact expectation      = {exact_expectation.real:.12f} + {exact_expectation.imag:.2e}j')
print(f'Taylor-step reference value = {reference_expectation.real:.12f} + {reference_expectation.imag:.2e}j')
print(f'absolute observable error   = {abs(reference_expectation - exact_expectation):.3e}')

dense sanity check: 100%|██████████| 1000000/1000000 [01:18<00:00, 12701.02it/s]

n_check = 8, r_check = 1000000
dense exact expectation      = -1.589558177183 + -3.47e-18j
Taylor-step reference value = -1.589526514389 + 6.94e-18j
absolute observable error   = 3.166e-05
